In [ ]:
# 구글드라이브 연동 그리고 깃허브 클론/풀
import os
import sys
import shutil
from google.colab import drive

drive.mount('/content/drive')

%cd /content
if os.path.exists('/content/korean-chatbot'):
    %cd korean-chatbot
    !git pull
else:
    !git clone https://github.com/kkkk2058/korean-chatbot.git
    %cd korean-chatbot

!pip install -r requirements.txt

In [ ]:
import shutil, os

os.makedirs("models", exist_ok=True)
shutil.copy("/content/drive/MyDrive/korean-chatbot/models/vocab.json", "models/vocab.json")

os.makedirs("data", exist_ok=True)
shutil.copy("/content/drive/MyDrive/korean-chatbot/data/namuwiki.txt", "data/namuwiki.txt")

print("파일 로드 완료!")
path = "data/namuwiki.txt"
print(f"namuwiki.txt 크기: {os.path.getsize(path) / 1024 / 1024:.1f} MB")

In [ ]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

In [ ]:
import sys
sys.path.append('/content/korean-chatbot')
from src.tokenizer import BPETokenizer
from src.model import Transformer

tok = BPETokenizer()
tok.load("models/vocab.json")

model = Transformer(vocab_size=tok.tokenizer.get_vocab_size())
model = model.to(device)
print(f"vocab size: {tok.tokenizer.get_vocab_size()}")
print(f"파라미터 수: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm  # 코랩 노트북 환경에 최적화된 tqdm 로드

# 0. 장치 정의 및 A100을 위한 TF32 연산 가속 활성화
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available() and "A100" in torch.cuda.get_device_name(0):
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    print("🚀 A100 GPU 감지: TF32 매트릭스 연산 가속을 켭니다.")

In [ ]:
# 1. 데이터셋 클래스
class TextDataset(Dataset):
    def __init__(self, path, tokenizer, max_seq_len=512):
        self.samples = []
        
        with open(path, "r", encoding="utf-8") as f:
            lines = [line.strip() for line in f if line.strip()]

        lines = lines

        encoded = tokenizer.tokenizer.encode_batch(lines)
        all_ids = []
        for e in encoded:
            all_ids.extend(e.ids)
        
        print(f"전체 토큰 수: {len(all_ids):,}")
        
        for i in range(0, len(all_ids) - max_seq_len, max_seq_len):
            self.samples.append(torch.tensor(all_ids[i:i+max_seq_len]))
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        return self.samples[idx]


# 2. 데이터로더 (★A100 GPU 병목 제거 최적화★)
dataset = TextDataset("data/namuwiki.txt", tok)

# A100의 거대한 VRAM을 활용하기 위해 batch_size를 64(또는 128)로 키우고,
# CPU 코어를 동원하는 num_workers와 메모리 고정(pin_memory)을 켜야 GPU가 놀지 않습니다.
dataloader = DataLoader(
    dataset, 
    batch_size=64,          
    shuffle=True,
    num_workers=4,          # CPU 데이터 준비 멀티프로세싱
    pin_memory=True,        # GPU 전송 고속 가속
    drop_last=True          # 연산 규격화를 위해 자투리 제거
)
print(f"총 샘플 수: {len(dataset):,}")
print(f"총 배치(스텝) 수: {len(dataloader):,}")

# 3. 옵티마이저
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

In [ ]:
EPOCHS = 10
model = model.to(device)

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    
    # dataloader를 tqdm으로 감싸서 화면에 예쁜 진행바를 띄웁니다.
    progress_bar = tqdm(
        enumerate(dataloader), 
        total=len(dataloader), 
        desc=f"Epoch {epoch+1}/{EPOCHS}"
    )
    
    for step, batch in progress_bar:
        batch = batch.to(device)
        
        # [A100 가속 핵심] bfloat16 자동 믹스드 프리시전(AMP)을 적용해 
        # 메모리를 절반만 쓰면서 연산 속도를 2~3배 끌어올립니다.
        with torch.amp.autocast(device_type='cuda', dtype=torch.bfloat16):
            loss = model.loss(batch)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        # 10스텝마다 tqdm 게이지바 우측에 실시간 Loss 상태를 표시합니다.
        # 지저분하게 print가 밑으로 늘어나는 것을 막아줍니다.
        if step % 10 == 0:
            progress_bar.set_postfix({
                "loss": f"{loss.item():.4f}",
                "avg_loss": f"{total_loss / (step + 1):.4f}"
            })
    
    avg_loss = total_loss / len(dataloader)
    print(f"✨ epoch {epoch+1} 완료 | 평균 loss: {avg_loss:.4f}\n")

print("🎉 모든 학습 완료!")




In [ ]:
import shutil

os.makedirs("models", exist_ok=True)
torch.save(model.state_dict(), "models/model.pt")

# Drive 백업
os.makedirs("/content/drive/MyDrive/korean-chatbot/models", exist_ok=True)
shutil.copy("models/model.pt", "/content/drive/MyDrive/korean-chatbot/models/model.pt")
print("모델 저장 & Drive 백업 완료!")



In [ ]:
model.eval()

test_inputs = [
    # 인사
    "안녕하세요",
    "반갑습니다",
    "좋은 아침이에요",
    
    # 한국 역사
    "조선은",
    "한국의 역사는",
    "고려시대에는",
    "삼국시대란",
    
    # 지식
    "인공지능이란",
    "머신러닝은",
    "딥러닝의 원리는",
    "트랜스포머 모델은",
    
    # 나무위키 스타일
    "대한민국은",
    "서울은",
    "한국어란",
    "김치는",
    
    # 문장 완성
    "오늘 날씨가",
    "밥을 먹으러",
    "학교에서",
    "회사에서",
]

for text in test_inputs:
    result = model.generate(text, tok)
    print(f"입력: {text}")
    print(f"출력: {result}")
    print("-" * 50)